# Módulo 16 · Práctica: leyes de escalado en miniatura

Los laboratorios no entrenan su modelo grande a ciegas: primero entrenan decenas de modelos pequeños, ajustan una **ley de potencia** y extrapolan. Aquí harás lo mismo a escala de portátil, con un lenguaje sintético cuya entropía conocemos exactamente.

| Parte | Qué haces |
| --- | --- |
| 0 | Preparación |
| 1 | Datos: deduplicación exacta y aproximada (MinHash) |
| 2 | Un lenguaje sintético con pérdida irreducible conocida |
| 3 | Entrenar 7 modelos de distinto tamaño |
| 4 | Ajustar leyes de potencia en log-log |
| 5 | Cómputo óptimo con la fórmula de Chinchilla |
| 6 | ¿Emergencia o espejismo de la métrica? |
| 7 | Retos |

Las celdas `# TU CÓDIGO` son ejercicios; la celda `# Comprobación` de detrás te dice si están bien. Todo corre en CPU en menos de 3 minutos.

## Parte 0 · Preparación

In [ ]:
import hashlib, re, time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
torch.set_num_threads(1)   # resultados reproducibles y sin pelear por la CPU
print(torch.__version__)

## Parte 1 · Deduplicación exacta y aproximada

Un rastreo web está lleno de copias: avisos legales, plantillas, noticias replicadas. Primero quitamos los duplicados **exactos** con un hash. Después buscamos **casi duplicados** con la similitud de Jaccard entre conjuntos de *shingles* (grupos de k palabras consecutivas):

$$J(A,B)=\frac{|A\cap B|}{|A\cup B|}$$

In [ ]:
docs = [
    "El gato duerme en el sofá del salón toda la tarde.",
    "El gato duerme en el sofá del salón toda la tarde.",
    "El gato duerme en el sofá del salón durante toda la tarde.",
    "Mañana lloverá en el norte y hará sol en el sur.",
    "Mañana lloverá en el norte y hará sol en todo el sur.",
    "La energía cinética de un cuerpo es un medio de su masa por la velocidad al cuadrado.",
    "Compra ya!!! Oferta oferta oferta oferta oferta",
    "Los modelos de lenguaje predicen el siguiente token a partir del contexto.",
]

def normalizar(t):
    return re.sub(r"[^\wáéíóúñü ]", "", t.lower()).split()

def shingles(t, k=3):
    w = normalizar(t)
    return {" ".join(w[i:i + k]) for i in range(len(w) - k + 1)}

print(shingles(docs[0]))

**Ejercicio 1.** Cuenta los documentos distintos usando `hashlib.sha1(d.encode()).hexdigest()` como huella. Guarda el número en `n_unicos`.

In [ ]:
# TU CÓDIGO
n_unicos = None

In [ ]:
# Comprobación
assert n_unicos == 7, "Debe haber 7 documentos distintos: solo el segundo es copia exacta del primero"
print("Correcto:", n_unicos)

**Ejercicio 2.** Escribe `jaccard(A, B)` para dos conjuntos.

In [ ]:
def jaccard(A, B):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
S = [shingles(d) for d in docs]
assert abs(jaccard(S[0], S[2]) - 7/12) < 1e-12, "J(doc0, doc2) debe ser 7/12"
assert jaccard(S[0], S[3]) == 0, "Documentos sin shingles comunes: J = 0"
print("Correcto: J(0,2) =", round(jaccard(S[0], S[2]), 4))

**MinHash.** Con n funciones hash hᵢ, la firma de un conjunto es (minₓ h₁(x), …, minₓ hₙ(x)). La probabilidad de que dos firmas coincidan en una posición es exactamente J(A, B). Así se estima J comparando firmas cortas, sin cruzar todos los shingles.

**Ejercicio 3.** Completa `firma` (usa la función `h(s, i)` ya dada) y estima J como la fracción de posiciones iguales.

In [ ]:
def h(s, semilla):
    return int.from_bytes(hashlib.blake2b(f"{semilla}|{s}".encode(), digest_size=8).digest(), "little")

In [ ]:
def firma(conjunto, n=128):
    # TU CÓDIGO: array de n mínimos, uno por semilla 0..n-1
    return None

def j_estimada(A, B, n=128):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
est = j_estimada(S[0], S[2])
assert firma(S[0]).shape == (128,), "La firma debe tener 128 componentes"
assert abs(est - 7/12) < 0.1, f"Estimación {est} demasiado lejos de 7/12"
assert j_estimada(S[0], S[3]) == 0
print("Correcto: J real 0,583 · J estimada", est)

**LSH por bandas.** Partimos la firma en b bandas de r filas; dos documentos son candidatos si coinciden en alguna banda entera. La probabilidad es $1-(1-s^r)^b$, una curva en S con umbral aproximado $(1/b)^{1/r}$.

**Ejercicio 4.** Implementa `p_candidato(s, r, b)`.

In [ ]:
def p_candidato(s, r, b):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
assert abs(p_candidato(0.5, 5, 20) - 0.4701) < 1e-3
assert abs(p_candidato(0.8, 5, 20) - 0.9996) < 1e-3
s = np.linspace(0, 1, 200)
plt.plot(s, p_candidato(s, 5, 20)); plt.axvline((1/20)**(1/5), ls="--", c="gray")
plt.xlabel("similitud de Jaccard s"); plt.ylabel("P(candidato)"); plt.title("LSH, r = 5, b = 20"); plt.show()
print("Correcto. Umbral ≈", round((1/20)**(1/5), 3))

## Parte 2 · Un lenguaje sintético con entropía conocida

Con texto real no sabemos cuánta pérdida es irreducible. Por eso inventamos un «lenguaje» de 32 tokens: el siguiente token depende de los **dos anteriores** según una tabla «profesora» fija. Los contextos siguen una ley de Zipf, como las palabras reales. La entropía condicional $E=H(X_t\mid X_{t-2},X_{t-1})$ es el suelo exacto de la pérdida: el término E de Chinchilla.

In [ ]:
V = 32; C = V * V
g = torch.Generator().manual_seed(0)
rango = torch.arange(1, C + 1).float()
p_ctx = rango ** -1.2; p_ctx /= p_ctx.sum()
p_ctx = p_ctx[torch.randperm(C, generator=g)]              # Zipf, en orden aleatorio
P_prof = F.softmax(2.0 * torch.randn(C, V, generator=g), 1)  # P(siguiente | contexto)
E_real = -(p_ctx[:, None] * P_prof * P_prof.log()).sum().item()
ctx = torch.arange(C); A_ctx, B_ctx = ctx // V, ctx % V
print(f"Pérdida irreducible E = {E_real:.4f} nats (uniforme: ln 32 = {np.log(32):.4f})")

# Un único flujo de datos de entrenamiento: 1.200 pasos × 256 ejemplos
PASOS, LOTE = 1200, 256
c = torch.multinomial(p_ctx, PASOS * LOTE, replacement=True, generator=g)
y = torch.multinomial(P_prof[c], 1, generator=g)[:, 0]
Atr, Btr, ytr = A_ctx[c].view(PASOS, LOTE), B_ctx[c].view(PASOS, LOTE), y.view(PASOS, LOTE)

## Parte 3 · Entrenar 7 modelos de tamaño creciente

Cada modelo es un MLP: embeddings de los dos tokens (d = 8), una capa oculta de anchura h con GELU y logits. Variamos h de 4 a 256. Guardamos la pérdida **exacta** (esperanza bajo el lenguaje verdadero, sin ruido de muestreo) tras ver D = 19.200; 38.400; …; 307.200 tokens.

In [ ]:
class MiniLM(nn.Module):
    def __init__(self, h, d=8):
        super().__init__()
        self.emb = nn.Embedding(V, d); self.l1 = nn.Linear(2 * d, h); self.l2 = nn.Linear(h, V)
    def forward(self, a, b):
        return self.l2(F.gelu(self.l1(torch.cat([self.emb(a), self.emb(b)], 1))))

def perdida_exacta(m):
    with torch.no_grad():
        return -(p_ctx[:, None] * P_prof * F.log_softmax(m(A_ctx, B_ctx), 1)).sum().item()

anchuras = [4, 8, 16, 32, 64, 128, 256]
controles = [75, 150, 300, 600, 1200]
resultados = []          # filas (N, D, L)
t0 = time.time()
for hh in anchuras:
    torch.manual_seed(0)
    m = MiniLM(hh); N = sum(p.numel() for p in m.parameters())
    opt = torch.optim.Adam(m.parameters(), lr=1e-2)
    for s in range(PASOS):
        loss = F.cross_entropy(m(Atr[s], Btr[s]), ytr[s])
        opt.zero_grad(); loss.backward(); opt.step()
        if s + 1 in controles:
            resultados.append((N, (s + 1) * LOTE, perdida_exacta(m)))
    print(f"h={hh:4d}  N={N:6d}  L final={resultados[-1][2]:.4f}  ({time.time()-t0:.0f} s)")
R = np.array(resultados)

In [ ]:
for D in np.unique(R[:, 1]):
    sel = R[:, 1] == D
    plt.loglog(R[sel, 0], R[sel, 2] - E_real, "o-", label=f"D = {int(D):,}".replace(",", "."))
plt.xlabel("parámetros N"); plt.ylabel("L − E (nats)"); plt.legend(); plt.title("Exceso de pérdida frente a N"); plt.show()

## Parte 4 · Ajustar una ley de potencia

Si $L-E = k\,N^{-\alpha}$, entonces $\log(L-E)=\log k-\alpha\log N$: una **recta** en log-log. Basta una regresión lineal (módulo 8).

**Ejercicio 5.** Con los modelos de D máximo, ajusta α y k con `np.polyfit` sobre los logaritmos.

In [ ]:
fin = R[R[:, 1] == R[:, 1].max()]
Nf, Lf = fin[:, 0], fin[:, 2]
# TU CÓDIGO
alfa, k = None, None

In [ ]:
# Comprobación
assert alfa is not None and 0.1 < alfa < 0.6, f"α = {alfa}: esperaba un exponente entre 0,1 y 0,6"
pred = E_real + k * Nf ** -alfa
assert np.max(np.abs(np.log(pred - E_real) - np.log(Lf - E_real))) < 0.2, "La recta no se ajusta bien"
print(f"Correcto: L ≈ {E_real:.3f} + {k:.2f}·N^(−{alfa:.3f})")
Ng = np.logspace(np.log10(Nf.min()), 5, 50)
plt.loglog(Nf, Lf - E_real, "o", label="medido"); plt.loglog(Ng, k * Ng ** -alfa, "--", label="ajuste y extrapolación")
plt.xlabel("N"); plt.ylabel("L − E"); plt.legend(); plt.show()

**Ajuste conjunto al estilo Chinchilla.** Ahora usamos toda la rejilla (N, D) y ajustamos $L=E+A/N^{\alpha}+B/D^{\beta}$ con E fijo, minimizando el error cuadrático de log L como hace el enfoque 3 del artículo (que usa además la pérdida de Huber). Lo hacemos con descenso de gradiente en PyTorch.

In [ ]:
Nt, Dt, Lt = (torch.tensor(R[:, i], dtype=torch.float64) for i in range(3))
par = torch.tensor([3.0, 0.5, 3.0, 0.3], dtype=torch.float64, requires_grad=True)  # log A, α, log B, β
opt = torch.optim.Adam([par], lr=0.02)
for it in range(3000):
    lA, a_, lB, b_ = par
    pred = E_real + torch.exp(lA - a_ * Nt.log()) + torch.exp(lB - b_ * Dt.log())
    err = ((pred.log() - Lt.log()) ** 2).mean()
    opt.zero_grad(); err.backward(); opt.step()
lA, a_, lB, b_ = par.detach().numpy()
print(f"A = {np.exp(lA):.1f}  α = {a_:.3f}  B = {np.exp(lB):.2f}  β = {b_:.3f}  (ECM en log: {err.item():.2e})")

Fíjate: el ajuste conjunto no tiene por qué dar el mismo α que la recta del ejercicio 5, porque ahí el término de datos B/D^β estaba mezclado con el de parámetros. Esta es una fuente real de discrepancia entre artículos (Kaplan frente a Chinchilla).

## Parte 5 · Cómputo óptimo con la fórmula de Chinchilla

Usamos el ajuste publicado (Hoffmann et al., 2022, enfoque 3): $E=1{,}69$, $A=406{,}4$, $B=410{,}7$, $\alpha=0{,}34$, $\beta=0{,}28$, y la restricción $C=6ND$. Con Lagrange se obtiene

$$N_{\rm opt}=G\left(\frac{C}{6}\right)^{\beta/(\alpha+\beta)},\qquad G=\left(\frac{\alpha A}{\beta B}\right)^{1/(\alpha+\beta)},\qquad D_{\rm opt}=\frac{C}{6N_{\rm opt}}.$$

**Ejercicio 6.** Implementa `optimo(C)` y compruébalo con una búsqueda por fuerza bruta.

In [ ]:
E_ch, A_ch, B_ch, a_ch, b_ch = 1.69, 406.4, 410.7, 0.34, 0.28
def L_ch(N, D):
    return E_ch + A_ch / N ** a_ch + B_ch / D ** b_ch

In [ ]:
def optimo(C):
    # TU CÓDIGO: devuelve (N_opt, D_opt)
    return None, None

In [ ]:
# Comprobación
C = 5.76e23                                   # presupuesto de Gopher y Chinchilla
Ns = np.logspace(9, 12, 300001); idx = np.argmin(L_ch(Ns, C / (6 * Ns)))
N_o, D_o = optimo(C)
assert abs(np.log(N_o / Ns[idx])) < 1e-3, "No coincide con la fuerza bruta"
assert abs(6 * N_o * D_o - C) / C < 1e-9, "Debe cumplirse C = 6ND"
print(f"Correcto: N_opt = {N_o:.3g}, D_opt = {D_o:.3g}, D/N = {D_o/N_o:.0f}, L = {L_ch(N_o, D_o):.4f}")
print(f"Chinchilla real (70e9, 1,4e12): L = {L_ch(70e9, 1.4e12):.4f} · Gopher (280e9, 300e9): L = {L_ch(280e9, 300e9):.4f}")

In [ ]:
Cs = np.logspace(19, 26, 50); No = np.array([optimo(c)[0] for c in Cs])
plt.loglog(Cs, No, label="ajuste enfoque 3"); plt.loglog(Cs, np.sqrt(Cs / 120), "--", label="regla D = 20N")
plt.xlabel("C (FLOP)"); plt.ylabel("N óptimo"); plt.legend(); plt.show()

**Ejercicio 7 · Sobreentrenar para abaratar la inferencia.** Con $C=10^{22}$ FLOP, toma la mitad del $N_{\rm opt}$. ¿Cuántos tokens $D_2$ necesitas para igualar la pérdida óptima? Calcula también el cómputo extra de entrenamiento `C2 / C`.

Pista: despeja D de $L^*=E+A/N^\alpha+B/D^\beta$.

In [ ]:
C = 1e22
N1, D1 = optimo(C); L1 = L_ch(N1, D1)
N2 = N1 / 2
# TU CÓDIGO
D2 = None
C2 = None

In [ ]:
# Comprobación
assert abs(L_ch(N2, D2) - L1) < 1e-9, "Con N2 y D2 debes obtener la misma pérdida"
assert abs(C2 / C - 1.208) < 0.005, f"C2/C = {C2/C:.3f}"
T = (C2 - C) / (2 * (N1 - N2))   # inferencia: 2N FLOP por token
print(f"Correcto: D2 = {D2:.3g} tokens ({D2/N2:.0f} por parámetro), C2/C = {C2/C:.3f}")
print(f"Se amortiza tras servir unos {T:.2g} tokens de inferencia")

## Parte 6 · ¿Emergencia o espejismo?

Supón que la probabilidad de acertar **cada token** de una respuesta mejora suavemente con la escala. Si la métrica exige acertar los k tokens a la vez (exact match), la exactitud es $p^k$: una curva que parece «despegar» de golpe. Es el argumento de Schaeffer et al. (2023).

**Ejercicio 8.** Calcula `exacta = p**k` para k = 10 y observa las dos curvas.

In [ ]:
logN = np.linspace(7, 12, 60)
p = 1 - 0.5 * 10 ** (-(logN - 7) / 3)   # mejora suave por token
k = 10
# TU CÓDIGO
exacta = None

In [ ]:
# Comprobación
assert exacta is not None and np.allclose(exacta, p ** 10)
plt.plot(logN, p, label="por token (suave)"); plt.plot(logN, exacta, label="respuesta exacta, k = 10")
plt.xlabel("log10 N (ficticio)"); plt.ylabel("exactitud"); plt.legend(); plt.show()
print("Correcto. Exactitud exacta: de", round(exacta[0], 4), "a", round(exacta[-1], 3))

## Parte 7 · Retos

1. **Ajuste en D.** Repite el ejercicio 5 con el modelo más grande, ajustando $L-E$ frente a D. ¿Qué β sale?
2. **Frontera de cómputo.** Para cada punto calcula $C=6ND$ y dibuja L frente a C. Traza la envolvente inferior: ¿qué tamaño es óptimo en cada presupuesto?
3. **Sin conocer E.** Ajusta también E. ¿Cuánto se aleja de `E_real`? ¿Por qué es difícil estimar la pérdida irreducible?
4. **Repetir datos.** Entrena el modelo de h = 256 con solo 150 lotes distintos repetidos 8 épocas. Compáralo con 1.200 lotes únicos (Muennighoff et al., 2023).

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# --- Ejercicio 1 ---
# n_unicos = len({hashlib.sha1(d.encode()).hexdigest() for d in docs})
# --- Ejercicio 2 ---
# def jaccard(A, B):
#     return len(A & B) / len(A | B)
# --- Ejercicio 3 ---
# def firma(conjunto, n=128):
#     return np.array([min(h(s, i) for s in conjunto) for i in range(n)], dtype=np.uint64)
#
# def j_estimada(A, B, n=128):
#     return np.mean(firma(A, n) == firma(B, n))
# --- Ejercicio 4 ---
# def p_candidato(s, r, b):
#     return 1 - (1 - np.asarray(s) ** r) ** b
# --- Ejercicio 5 ---
# fin = R[R[:, 1] == R[:, 1].max()]
# Nf, Lf = fin[:, 0], fin[:, 2]
# pend, corte = np.polyfit(np.log(Nf), np.log(Lf - E_real), 1)
# alfa, k = -pend, np.exp(corte)
# --- Ejercicio 6 ---
# def optimo(C):
#     G = (a_ch * A_ch / (b_ch * B_ch)) ** (1 / (a_ch + b_ch))
#     N = G * (C / 6) ** (b_ch / (a_ch + b_ch))
#     return N, C / (6 * N)
# --- Ejercicio 7 ---
# C = 1e22
# N1, D1 = optimo(C); L1 = L_ch(N1, D1)
# N2 = N1 / 2
# D2 = (B_ch / (L1 - E_ch - A_ch / N2 ** a_ch)) ** (1 / b_ch)
# C2 = 6 * N2 * D2
# --- Ejercicio 8 ---
# logN = np.linspace(7, 12, 60)
# p = 1 - 0.5 * 10 ** (-(logN - 7) / 3)
# k = 10
# exacta = p ** k